# NAV Spike Auditor

This notebook explains **why your live/backtest NAV suddenly jumps or collapses**.

It reads a finished run from `runs/<run_id>/events.jsonl`, finds the largest NAV moves,
and then attributes each move back to the symbols you were holding at that moment.

---

## What this notebook is trying to answer

When you see a cliff in the NAV chart, there are usually only a few possibilities:

1. a held symbol got a **bad price**
2. a held symbol underwent an **unadjusted split/bonus-like scale change**
3. the strategy traded around the event and locked in the damage

This notebook focuses on the first two by estimating, for each spike:

\[
\Delta NAV pprox \sum_{s \in 	ext{held}} qty_s \cdot (p_{t,s} - p_{t-1,s})
\]

That is enough to identify which symbols are driving the move.


## Inputs you need

- `RUN_DIR`: a completed run directory containing `events.jsonl`
- `STORE_DIR`: the matrix store used by `MatrixStoreMinuteFeed` or its repaired version

The audit output is saved into a folder inside the run directory so you can keep the
explanation next to the run that generated the suspicious NAV behaviour.


In [1]:
from pathlib import Path
import pandas as pd

from analytics.nav_spike_audit import audit_nav_spikes, save_nav_spike_audit

RUN_DIR = Path('runs/CHANGE_ME_RUN_ID')
STORE_DIR = Path('CHANGE_ME_MATRIX_STORE_DIR')

# A 5% minute-to-minute NAV move is already huge for a diversified portfolio.
PCT_NAV_CHANGE = 0.05
TOP_K_SYMBOLS = 10


## Run the audit

The auditor will:

1. load `position_snapshot` events from the run
2. find the large NAV jumps
3. reconstruct **as-of prices** from the matrix store on both sides of each jump
4. rank the symbols by estimated NAV contribution


In [2]:
result = audit_nav_spikes(
    run_dir=RUN_DIR,
    store_dir=STORE_DIR,
    pct_nav_change=PCT_NAV_CHANGE,
    top_k_symbols=TOP_K_SYMBOLS,
)

output_dir = RUN_DIR / 'nav_spike_audit'
spikes_path, contrib_path = save_nav_spike_audit(result, output_dir)

print('Saved spikes table to:', spikes_path)
print('Saved contribution table to:', contrib_path)
print('Number of spikes found:', len(result.spikes))


FileNotFoundError: [Errno 2] No such file or directory: 'runs\\CHANGE_ME_RUN_ID\\events.jsonl'

## Read the headline table first

Start with the `spikes` table. The most useful columns are:

- `ts_prev`, `ts_now`: the two timestamps around the jump
- `nav_change`, `nav_change_pct`: size of the move
- `top_symbol`: the symbol that contributed the most to the estimated move
- `estimated_explained_change`: how much of the NAV move was explained by available prices

If `symbols_with_missing_prices` is large, the explanation is incomplete and you should
inspect the store around that timestamp more carefully.


In [ ]:
pd.set_option('display.max_columns', 50)
result.spikes.head(20)


## Drill into one suspicious event

Pick a `spike_id` from the contributions table and inspect the largest symbol-level
contributors. In practice, one or two symbols usually explain most fake NAV cliffs.


In [ ]:
if not result.contributions.empty:
    spike_id = int(result.contributions.iloc[0]['spike_id'])
    display(result.contributions[result.contributions['spike_id'] == spike_id].head(20))
else:
    print('No symbol-level contributions were produced.')


## How to use the result

- If the same symbol shows up repeatedly, repair that symbol in the store.
- If many different symbols show up with single-bar jumps, turn on the **sanitized** feed.
- If the move happens exactly when the strategy traded, inspect fills as well; some of the
  damage may already be realized in the portfolio rather than only in the mark-to-market.
